# 🚀 LifeMate Agent — Colab / Kaggle Runner & Submission Generator
### Zalo AI Challenge 2026 — Voice-driven AI Agent

Notebook này cho phép bạn chạy toàn bộ luồng xử lý trên **Google Colab (GPU T4 miễn phí)** hoặc **Kaggle (2x T4)**:
1. **Part B (ASR)**: `faster-whisper large-v3-turbo` chuyển đổi giọng nói tiếng Việt thành văn bản.
2. **Part C (Agent LLM)**: `Qwen/Qwen3-4B-Instruct-2507` (4.02B params) suy luận và gọi công cụ (Tool Calling).
3. **Part A (Lead/Eval)**: Quản lý ToolEnv, trích xuất `tool_calls` & `state_hash`, kiểm tra 7 tầng bằng `SubmissionLinter` và xuất file nộp bài chuẩn.

## 1. Kiểm tra GPU & Kết nối Google Drive

In [ ]:
# 1. Kiểm tra GPU T4 (15-16GB VRAM)
!nvidia-smi

# 2. Kết nối Google Drive để lưu bài nộp và checkpoint
from google.colab import drive
drive.mount('/content/drive')

## 2. Cài đặt các thư viện phụ thuộc

In [ ]:
!pip install -q transformers>=4.45 accelerate torch faster-whisper sentencepiece huggingface_hub jiwer

## 3. Clone / Cập nhật mã nguồn & Chuẩn bị dữ liệu

In [ ]:
import os
from pathlib import Path

# 1. Clone repository nếu chưa có trên Colab SSD
WORKSPACE = Path("/content/LifeMateAgent")
if not WORKSPACE.exists():
    !git clone https://github.com/Platypus27-coder/zalo-ai-challenge.git /content/LifeMateAgent

%cd /content/LifeMateAgent
!git checkout main
!git pull origin main

# 2. Giải nén dữ liệu public_test hoặc train+public_test từ Google Drive
DATA_DIR = WORKSPACE / "train+public_test"
DRIVE_PUB_ZIP = Path("/content/drive/MyDrive/public_test.zip")
DRIVE_FULL_ZIP = Path("/content/drive/MyDrive/train+public_test.zip")

if not (DATA_DIR / "public_test" / "public_test.jsonl").exists():
    if DRIVE_PUB_ZIP.exists():
        print(f"📦 Đang giải nén dữ liệu public_test từ: {DRIVE_PUB_ZIP}...")
        !unzip -q -o "{DRIVE_PUB_ZIP}" -d /content/LifeMateAgent/
    elif DRIVE_FULL_ZIP.exists():
        print(f"📦 Đang giải nén dữ liệu đầy đủ từ: {DRIVE_FULL_ZIP}...")
        !unzip -q -o "{DRIVE_FULL_ZIP}" -d /content/LifeMateAgent/
    else:
        print("⚠️ Không tìm thấy file zip dữ liệu tại Google Drive.")

if (DATA_DIR / "public_test" / "public_test.jsonl").exists():
    print("\n✅ Dữ liệu public_test (634 mẫu) đã sẵn sàng!")
else:
    print("\n❌ Chưa có thư mục train+public_test/public_test/")

## 4. Kiểm tra hợp chuẩn toàn bộ hệ thống (Self-Verification)

In [ ]:
!python scripts/verify_all.py

## 5. Chạy thử nghiệm khói (Smoke Test: 5 mẫu Public Test)
Chạy thử 5 mẫu đầu tiên để xác nhận mô hình ASR (`faster-whisper large-v3-turbo`) và Agent LLM (`Qwen3-4B-Instruct-2507`) nạp vào GPU T4 mượt mà, không gặp lỗi bộ nhớ hay định dạng.

In [ ]:
!python scripts/generate_submission.py \
    --split public_test \
    --max-samples 5 \
    --asr-mode whisper \
    --asr-model large-v3-turbo \
    --agent-mode transformers \
    --model-name Qwen/Qwen3-4B-Instruct-2507 \
    --verbose

## 6. Chạy chính thức toàn bộ 634 mẫu Public Test & Tạo File Nộp Bài
Sau khi smoke test thành công, chạy toàn bộ tập `public_test`. Script sẽ tự động:
1. Phiên âm audio bằng faster-whisper.
2. Cho Qwen3-4B phân tích ngữ cảnh, gọi tool và lưu trạng thái vào ToolEnv.
3. Xuất file `submission.jsonl` và tự động kiểm tra 7 tầng quy chuẩn của Zalo AI Challenge qua `SubmissionLinter`.

In [ ]:
# Tạo thư mục chứa bài nộp
!mkdir -p /content/outputs

!python scripts/generate_submission.py \
    --split public_test \
    --asr-mode whisper \
    --asr-model large-v3-turbo \
    --agent-mode transformers \
    --model-name Qwen/Qwen3-4B-Instruct-2507 \
    --output /content/outputs/submission_public_test.jsonl \
    --verbose

## 7. Sao lưu vĩnh viễn lên Google Drive & Tải về máy tính
Lưu bài nộp vào Google Drive để không bị mất khi đóng tab, và tự động tải file nộp bài về máy để upload lên Leaderboard của Zalo AI Challenge.

In [ ]:
import shutil
from google.colab import files
from pathlib import Path

sub_file = Path("/content/outputs/submission_public_test.jsonl")
if sub_file.exists():
    # 1. Lưu vào Drive
    drive_dest = Path("/content/drive/MyDrive/LifeMate_Submissions")
    drive_dest.mkdir(parents=True, exist_ok=True)
    shutil.copy(sub_file, drive_dest / sub_file.name)
    print(f"✅ Đã sao lưu vào Google Drive: {drive_dest / sub_file.name}")
    
    # 2. Tải về máy tính qua trình duyệt
    print("⬇️ Đang tải file về máy tính của bạn...")
    files.download(str(sub_file))
else:
    print("❌ Không tìm thấy file bài nộp:", sub_file)